## Necessary import and defination

In [1]:
import cupy as cp
cp.cuda.Device(2).use()
import os
## change the directory to the wholistic_registration code dir, so that we can import the modules in utils
os.chdir('/home/cyf/wbi/Virginia/code/wbi_0123/wholistic_registration/src/wholistic_registration')
from utils import IO
from utils import calFlowCrossResolution, calFlow3d_Wei_v1
from utils import registration
from utils import option
from utils import preprocess as prep
from utils import mask
from utils import  visualization

F260517_mov_path = "/home/cyf/wbi/Virginia/raw_data/f260517/260517_exp_00001_TZCYX.ome.tiff"
F260517_ref_path = "/home/cyf/wbi/Virginia/raw_data/f260517/260517_anat_00003_TZCYX.ome.tiff"

CuPy is available with CUDA - using GPU acceleration


zRatio = 10, frame_rate = 4.07066 s/frame

Test to do the registration
---

In [ ]:
from importlib import reload
from typing import Any

from numpy import ndarray
import numpy as np

reload(calFlowCrossResolution)
reload(prep)

F260517_mov, F260517_mov_desc = IO.readTiff(F260517_mov_path)
F260517_ref, F260517_ref_desc = IO.readTiff(F260517_ref_path)
print("Finish reading the dataset")


########### Split into 2 stacks
F260517_ref_mem = F260517_ref[90:310, 1, :, :]
F260517_ref_sparseCell = F260517_ref[90:310, 0, :, :]

F260517_mov_mem: ndarray[tuple[int, ...], Any] = F260517_mov[:, :, 1, :, :]
F260517_mov_sparseCell: ndarray[tuple[int, ...], Any] = F260517_mov[:, :, 0, :, :]

print("F260517_ref_mem:", F260517_ref_mem.shape)          # (Zref, Y, X)
print("F260517_mov_mem:", F260517_mov_mem.shape)          # (T, K, Y, X)
print("Finish splitting the dataset")


########### Registration options
option['r'] = 5
option['layer'] = 3
option['iter'] = 10
option['movRange'] = 5.
option['tol'] = 1e-6
option['zRatio_HR'] = 1
option["wrong_region_enable"] = False

thresFactor = 5.
maskRange = [5., 4000.]
smoothPenalty_raw = 0.01


########### Find the corresponding z slice
z_init = calFlowCrossResolution.FindInitZ_stack_global_fixed_spacing(
    F260517_mov_mem[0].transpose(2, 1, 0),     # (X, Y, K)
    F260517_ref_mem.transpose(2, 1, 0),        # (X, Y, Zref)
    delta_ref_idx=10,
    use_gradient=False
)

z_init = np.asarray(z_init, dtype=np.float32)
z_idx = np.rint(z_init).astype(np.int32)
z_idx = np.clip(z_idx, 0, F260517_ref_mem.shape[0] - 1)

K = z_init.shape[0]

x = np.arange(F260517_mov_mem[0].shape[2], dtype=np.float32)
y = np.arange(F260517_mov_mem[0].shape[1], dtype=np.float32)
k = np.arange(K, dtype=np.int32)

X_grid, Y_grid, K_grid = np.meshgrid(
    x,
    y,
    k,
    indexing="ij",
)

coords_xyz = np.empty(
    (F260517_mov_mem[0].shape[2], F260517_mov_mem[0].shape[1], K, 3),
    dtype=np.float32
)

coords_xyz[..., 0] = X_grid
coords_xyz[..., 1] = Y_grid
coords_xyz[..., 2] = z_init[K_grid]

option["phase"] = coords_xyz

print("z_init:", z_init)
print("z_idx:", z_idx)
print("phase shape:", option["phase"].shape)


########### Quantile mapping helper
percentiles = [
    0.1, 0.5, 1, 2, 5,
    10, 25, 50, 75, 90,
    95, 99, 99.5, 99.8
]


def update_reference_intensity_mapping(
    F260517_ref_mem,
    target_mov_zyx,
    z_idx,
    option,
    thresFactor,
    maskRange,
    smoothPenalty_raw,
    percentiles,
):
    """
    Learn ref -> target_mov intensity mapping and update:
        1. F260517_ref_mem_adj
        2. option['mask_ref']
        3. option['smoothPenalty']

    F260517_ref_mem: (Zref, Y, X)
    target_mov_zyx:  (K, Y, X)
    z_idx:           (K,)
    """

    target_mov_zyx = np.asarray(target_mov_zyx, dtype=np.float32)

    if target_mov_zyx.ndim != 3:
        raise ValueError(f"target_mov_zyx should be (K,Y,X), got {target_mov_zyx.shape}")

    if target_mov_zyx.shape[0] != len(z_idx):
        raise ValueError(
            f"target_mov_zyx K={target_mov_zyx.shape[0]} does not match z_idx length={len(z_idx)}"
        )

    ref_source_zyx = F260517_ref_mem[z_idx]  # (K, Y, X)

    src_q, tgt_q, used_percentiles = prep.learn_quantile_mapping(
        source=ref_source_zyx,
        target=target_mov_zyx,
        percentiles=percentiles,
    )

    F260517_ref_mem_adj = prep.apply_quantile_mapping(
        F260517_ref_mem,
        src_q,
        tgt_q
    ).transpose(2, 1, 0).astype(np.float32, copy=False)  # (X, Y, Zref)

    option['mask_ref'] = mask.getMask(F260517_ref_mem_adj, thresFactor)
    option['mask_ref'] = mask.bwareafilt3_wei(option['mask_ref'], maskRange)

    Pnltfactor = prep.getSmPnltNormFctr(F260517_ref_mem_adj, option)
    option['smoothPenalty'] = Pnltfactor * smoothPenalty_raw

    return F260517_ref_mem_adj, src_q, tgt_q, used_percentiles


########### Initial reference adjustment using frame 0
F260517_ref_mem_adj, src_q, tgt_q, used_percentiles = update_reference_intensity_mapping(
    F260517_ref_mem=F260517_ref_mem,
    target_mov_zyx=F260517_mov_mem[0],
    z_idx=z_idx,
    option=option,
    thresFactor=thresFactor,
    maskRange=maskRange,
    smoothPenalty_raw=smoothPenalty_raw,
    percentiles=percentiles,
)

print("[Init] Updated reference intensity mapping using frame 0")
print("src_q:", src_q)
print("tgt_q:", tgt_q)


########### Generate H for sparseCell reference
H_sparseCell = calFlowCrossResolution.generate_continuous_H_gpu(
    F260517_ref_sparseCell.transpose(2, 1, 0),
    zRatio=1
)


########### Reference update setting
update_every = 40
update_mode = "single_frame"


########### Registration
T = F260517_mov_mem.shape[0]

for i in range(T):

    if i > 75:
        option['wrong_region_enable']=True
    F260517_mov_mem_i = F260517_mov_mem[i].transpose(2, 1, 0)              # (X, Y, K)
    F260517_mov_sparseCell_i = F260517_mov_sparseCell[i].transpose(2, 1, 0)

    option['mask_mov'] = mask.getMask(F260517_mov_mem_i, thresFactor)
    option['mask_mov'] = mask.bwareafilt3_wei(option['mask_mov'], maskRange)

    phase_new, motion_current, mem_mapped = calFlowCrossResolution.getMotion_v2(
        F260517_mov_mem_i,
        F260517_ref_mem_adj,
        option,
        verbose=False
    )

    sparseCell_mapped = calFlowCrossResolution.apply_H_to_matrix_gpu(
        phase_new,
        H_sparseCell
    ).get()

    print(f"Frame {i}")
    print("Mem channel error: ", np.mean(np.abs(F260517_mov_mem_i - mem_mapped)))
    print("Sparse cell channel error: ", np.mean(np.abs(F260517_mov_sparseCell_i - sparseCell_mapped)))

    option['motion'] = 0.7 * motion_current

    IO.write_multichannel_volume_as_ome_tiff(
        volume=[
            F260517_mov_mem_i[::2, ::2, :].transpose(2, 1, 0),
            mem_mapped[::2, ::2, :].transpose(2, 1, 0),
            F260517_mov_sparseCell_i[::2, ::2, :].transpose(2, 1, 0),
            sparseCell_mapped[::2, ::2, :].transpose(2, 1, 0),
        ],
        out_dir="/home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0519_v2/",
        ######  the output folder
        frame_idx=i,
        label='F260517'
    )
    should_update_ref = ((i + 1) % update_every == 0) and ((i + 1) < T)

    if should_update_ref:
        if update_mode == "single_frame":
            # shape: (K, Y, X)
            target_mov_for_update = F260517_mov_mem[i].astype(np.float32)

            update_info = f"single frame {i}"

        elif update_mode == "mean_last5":
            start_idx = max(0, i - 4)
            end_idx = i + 1

            target_mov_for_update = np.mean(
                F260517_mov_mem[start_idx:end_idx].astype(np.float32),
                axis=0
            )

            update_info = f"mean raw mov frames {start_idx}-{end_idx - 1}"

        else:
            raise ValueError(f"Unknown update_mode: {update_mode}")

        F260517_ref_mem_adj, src_q, tgt_q, used_percentiles = update_reference_intensity_mapping(
            F260517_ref_mem=F260517_ref_mem,
            target_mov_zyx=target_mov_for_update,
            z_idx=z_idx,
            option=option,
            thresFactor=thresFactor,
            maskRange=maskRange,
            smoothPenalty_raw=smoothPenalty_raw,
            percentiles=percentiles,
        )

        print("=" * 80)
        print(f"[After frame {i}] Updated reference intensity mapping using {update_info}")
        print("src_q:", src_q)
        print("tgt_q:", tgt_q)
        print("=" * 80)

Finish reading the dataset
F260517_ref_mem: (220, 1500, 630)
F260517_mov_mem: (200, 20, 1500, 630)
Finish splitting the dataset
z_init: [ 20.  30.  40.  50.  60.  70.  80.  90. 100. 110. 120. 130. 140. 150.
 160. 170. 180. 190. 200. 210.]
z_idx: [ 20  30  40  50  60  70  80  90 100 110 120 130 140 150 160 170 180 190
 200 210]
phase shape: (630, 1500, 20, 3)
[Init] Updated reference intensity mapping using frame 0
src_q: [ -188.     -156.     -140.     -120.      -87.      -53.       26.
   314.     1897.     4546.     5872.     8854.    10194.    11971.002]
tgt_q: [-195. -170. -157. -141. -114.  -86.  -23.  158. 1072. 2578. 3352. 5064.
 5848. 6917.]
Frame 0
Mem channel error:  172.50751203822367
Sparse cell channel error:  36.11024259259259
Frame 1
Mem channel error:  172.7088268365443
Sparse cell channel error:  35.13069
Frame 2
Mem channel error:  173.0980710162898
Sparse cell channel error:  36.75155465608466
Frame 3
Mem channel error:  172.82023464395905
Sparse cell channel error: